# Lecture 2: Unix Workflows, Automation & Reproducibility

**Course:** Computational Physics & Scientific Programming  
**Scheduled lecture time:** 2.5 hours  

This notebook is a **deep, hands-on laboratory** for Unix workflows used in Computational Physics.

---

## How to use this notebook
Focus on sections:
1) Standard streams + redirection + pipes (Sections 1–3)  
2) Data inspection discipline (Section 4)  
3) One workflow mini-project (Section 10)  



**Rule of the week:**  
> If your work is not scriptable, it is not reproducible.

---

## What you will build by the end
A **mini workflow system** that can:
- generate "dummy" datasets (like simulation outputs)  
- log runs in a reproducible way  
- compute quick diagnostics from the terminal  
- pass data to Python for plotting hooks  
- keep results organized (so you can reproduce them later)


## Setup: Create a clean workspace

We will work inside a dedicated folder so that files remain organized.

**Folder structure (recommended):**
- `data/`  (raw and derived datasets)  
- `scripts/`  (shell scripts and small utilities)  
- `results/`  (logs, summaries, plots)  
- `notes/`  (reflection notes, experiment notes)  

Run the next cell to create this structure.


In [ ]:
%%bash
set -e
mkdir -p lecture2/{data,scripts,results,notes}
cd lecture2
pwd
find . -maxdepth 2 -type d | sort


---
# Standard Streams: stdin, stdout, stderr

Unix commands communicate through **streams**:
- **stdin** (input)
- **stdout** (normal output)
- **stderr** (errors)

### Why physicists care
In simulations, you usually want:
- results to go to files (stdout redirected)
- warnings/errors logged separately (stderr redirected)
- pipelines connecting tools (stdout → stdin)

We will practice these *as if running experiments*.


### A. stdout: printing normal output

In [ ]:
%%bash
set -e
cd lecture2
echo "Run metadata: dt=0.01, N=1000"

### B. stderr: intentionally cause an error and observe the message

In [ ]:
%%bash
set -e
cd lecture2
ls this_folder_does_not_exist


### C. Redirect stdout vs stderr (explicit)

- `>` redirects **stdout**  
- `2>` redirects **stderr**  
- `&>` redirects both (bash)

Try these patterns below.


In [ ]:
%%bash
set -e
cd lecture2
# stdout to file
echo "Simulation started" > results/stdout.log

# stderr to file (will write an error message)
ls missing_dir 2> results/stderr.log || true

echo "---- stdout.log ----"
cat results/stdout.log
echo "---- stderr.log ----"
cat results/stderr.log


### Trial Question
1) Redirect both stdout and stderr to a single file called `results/all.log`  
2) Use `cat` to show it  
3) Explain what you see (write 2–3 sentences in your notes)

**Hint:** `&>` or `> file 2>&1`


In [ ]:
%%bash
set -e
cd lecture2
# Demo of redirecting both streams
{
  echo "This is stdout"
  ls missing_dir
} &> results/all.log || true

echo "---- all.log ----"
cat results/all.log


---
# Redirection: keeping scientific records

Redirection turns terminal output into **persistent scientific records**.

Key patterns:
- `>` overwrite (new experiment record)
- `>>` append (experiment log book)
- `tee` write to screen **and** file

In real work, you often want:
- a run log per experiment
- a master log for the entire project


### A. Overwrite vs append

In [ ]:
%%bash
set -e
cd lecture2
echo "Run 1: dt=0.1" > results/runbook.log
echo "Run 2: dt=0.05" >> results/runbook.log
echo "Run 3: dt=0.02" >> results/runbook.log
cat results/runbook.log

### B. Use `tee` for live logging

In [ ]:
%%bash
set -e
cd lecture2
echo "A message that appears on screen AND in file" | tee results/tee_demo.log
cat results/tee_demo.log


### C. Logging with timestamps (essential discipline)

We will build a runbook entry format:
- ISO timestamp
- short message
- key parameter=value pairs

We will use `date` to generate timestamps.


In [ ]:
%%bash
set -e
cd lecture2
ts=$(date -Iseconds)
echo "$ts | EVENT=START | dt=0.01 | N=1000" >> results/runbook.log
tail -n 3 results/runbook.log


### Try it yourself (Redirection + logging)
Add three entries to `results/runbook.log`:
1) START with dt=0.02 N=500  
2) NOTE with message="testing stability"  
3) END with status=OK  


---
# Pipes: building workflows (from single commands to pipelines)

A pipe connects:
`stdout` of one command → `stdin` of another command

This is *exactly* like chaining transformations.

We will build pipelines for:
- counting
- filtering
- sorting
- extracting maxima/minima
- generating summaries


### A. Warm-up pipelines

In [ ]:
%%bash
set -e
cd lecture2
ls | wc -l
find . -type f | wc -l


### B. Create a small CSV dataset (`values.csv`)

In [ ]:
%%bash
set -e
cd lecture2
cat > data/values.csv << 'EOF'
t,x
0,0
1,1
2,4
3,9
4,16
5,25
EOF

cat data/values.csv


### C. Extract a column, skip header, sort, compute min/max

In [ ]:
%%bash
set -e
cd lecture2
# Extract x column (field 2), skip header, sort numerically
cut -d, -f2 data/values.csv | tail -n +2 | sort -n > results/x_sorted.txt

echo "Sorted x values:"
cat results/x_sorted.txt

echo "Min:"
head -n 1 results/x_sorted.txt

echo "Max:"
tail -n 1 results/x_sorted.txt


### Try it yourself (Pipelines)
1) Create `data/values2.csv` with 20 rows, where x is not sorted  
2) Use a pipeline to compute the max x  
3) Save your pipeline output to `results/max_x.txt`  


---
# Iterations (Loops) in Unix: Repeating Work Safely and Reproducibly

In Computational Physics, almost everything involves **iteration**:
- time-stepping (repeat an update rule many times)
- parameter sweeps (repeat a run for many parameter values)
- batch processing many files (repeat the same transformation on many datasets)

Unix provides iteration through:
- `for` loops
- `while` loops
- `xargs` and `find ... -exec` patterns

**Key goal:** build iterations that are **safe**, **logged**, and **re-runnable**.

> A good loop produces the same outputs every time, and leaves evidence of what it did.

---

## Core ideas (in increasing strength order)

1) **Loop over a known set** (files or parameter values)  
2) **Store outputs per iteration** (never overwrite silently)  
3) **Log each iteration** (timestamp + parameters)  
4) **Fail safely** (capture failures and continue or stop intentionally)  
5) **Scale** (replace manual loops with structured sweeps / batch tools)

---


## A `for` loops: repeat over a fixed set

`for` loops are ideal when you know what you want to loop over:
- a list of dt values
- a list of filenames
- a list of experiment labels

### Pattern
```bash
for x in a b c; do
  command "$x"
done
```


In [ ]:
%%bash
set -e
cd lecture2

# Example: loop over dt values (fixed set)
for dt in 0.2 0.1 0.05 0.02 0.01; do
  echo "dt=$dt"
done


### Try it yourself (for loops)
1) Create a folder `results/runs/`  
2) For each dt in `0.2 0.1 0.05 0.02 0.01`, create a file named `run_dt_<dt>.txt`  
3) Put a single line in each file: `dt=<dt>`  


In [ ]:
%%bash
set -e
cd lecture2
mkdir -p results/runs

for dt in 0.2 0.1 0.05 0.02 0.01; 
do
  echo "dt=$dt" > "results/runs/run_dt_${dt}.txt"
done

ls -1 results/runs | head
echo "Sample file content:"
cat results/runs/run_dt_0.1.txt


## B Logging each iteration (runbook discipline)

In research, every iteration should leave a trace:
- timestamp
- parameter values
- output file locations
- status (OK/FAIL)

We'll append iteration logs to `results/iter_runbook.log`.


In [ ]:
%%bash
set -e
cd lecture2
: > results/iter_runbook.log

for dt in 0.2 0.1 0.05 0.02 0.01; do
  ts=$(date -Iseconds)
  echo "$ts | ITER dt=$dt | ACTION=WRITE_FILE | out=results/runs/run_dt_${dt}.txt | status=OK" >> results/iter_runbook.log
done

tail -n 5 results/iter_runbook.log


## C `while` loops: repeat until a condition is met

`while` loops are useful when:
- you don't know the number of iterations ahead of time
- you stop based on an event (e.g., convergence, a threshold, end-of-file)

### Pattern
```bash
while condition; do
  command
done
```
We will use `while read` to process lines from a file (very common for parameter lists).


In [ ]:
%%bash
set -e
cd lecture2

# Process dt values from file (line-by-line)
echo "Reading dt values from data/params_dt.txt:"
while read -r dt; do
  echo "dt=$dt"
done < data/params_dt.txt


### Try it yourself (while loops)
Modify the loop so that it:
- skips empty lines
- ignores lines starting with `#` (comments)


In [ ]:
%%bash
set -e
cd lecture2_18h

# Make a parameter file with comments and blanks
cat > data/params_dt_commented.txt << 'EOF'
# dt list for stability experiments

0.2
0.1

# refined steps
0.05
0.02
0.01
EOF

echo "Clean-reading dt values (skip blanks/comments):"
while read -r line; do
  [[ -z "$line" ]] && continue
  [[ "$line" =~ ^# ]] && continue
  echo "dt=$line"
done < data/params_dt_commented.txt


## D Iteration with failure handling (continue vs stop)

In experiments, some runs fail. You must decide:
- **stop immediately** (strict mode)  
- **log and continue** (sweep mode)  

We'll demonstrate both.


In [ ]:
%%bash
set -e
cd lecture2
: > results/iter_fail_demo.log

echo "Mode A: stop on first failure (strict)"
set +e
for dt in 0.2 0.1 0.05; do
  ./scripts/fake_sim.sh "$dt" 100 >> results/iter_fail_demo.log 2>&1
  code=$?
  if [[ $code -ne 0 ]]; then
    echo "$(date -Iseconds) | dt=$dt | status=FAIL | exit_code=$code | STOP" >> results/iter_fail_demo.log
    break
  else
    echo "$(date -Iseconds) | dt=$dt | status=OK" >> results/iter_fail_demo.log
  fi
done
set -e

tail -n 25 results/iter_fail_demo.log


In [ ]:
%%bash
set -e
cd lecture2_18h
: > results/iter_fail_demo_continue.log

echo "Mode B: log failures and continue (sweep)"
set +e
for dt in 0.2 0.1 0.05 0.02 0.01; do
  ./scripts/fake_sim.sh "$dt" 100 >> results/iter_fail_demo_continue.log 2>&1
  code=$?
  if [[ $code -ne 0 ]]; then
    echo "$(date -Iseconds) | dt=$dt | status=FAIL | exit_code=$code | CONTINUE" >> results/iter_fail_demo_continue.log
  else
    echo "$(date -Iseconds) | dt=$dt | status=OK" >> results/iter_fail_demo_continue.log
  fi
done
set -e

tail -n 25 results/iter_fail_demo_continue.log


## E Iteration over many files: `find` and `xargs` patterns

When you have **many output files**, you often want to apply an action to all of them:
- count lines
- compute checksums
- compress
- move into archives

### `find ... -exec` (safe)
```bash
find results/runs -type f -name "*.txt" -exec wc -l {} \;
```

### `xargs` (fast; be careful with spaces)
```bash
find results/runs -type f -name "*.txt" | xargs wc -l
```

We'll practice both. Use the `-print0` / `-0` form for safety.


In [ ]:
%%bash
set -e
cd lecture2_18h

echo "find -exec:"
find results/runs -type f -name "*.txt" -exec wc -l {} \; | head

echo ""
echo "xargs (safe -0):"
find results/runs -type f -name "*.txt" -print0 | xargs -0 wc -l | head


### Reflection (Iterations)
Answer in `notes/iterations_reflection.md`:
1) When would you use `for` vs `while` in scientific workflows?  
2) Why is logging per iteration important for reproducibility?  
3) Give one real physics example where a failed iteration could mislead results if unlogged.  


In [ ]:
%%bash
set -e
cd lecture2_18h
cat > notes/iterations_reflection.md << 'EOF'
# Iterations Reflection

1) When would you use `for` vs `while` in scientific workflows?

2) Why is logging per iteration important for reproducibility?

3) Give one real physics example where a failed iteration could mislead results if unlogged.

EOF
sed -n '1,120p' notes/iterations_reflection.md


---
# Inspect before you analyze (data hygiene)

Never trust data you have not inspected.

Essential commands:
- `wc -l` (how many rows?)
- `head`, `tail` (sanity check)
- `sort`, `uniq` (duplicates, ordering)
- `grep` (anomalies, patterns)

We practice these like laboratory checks.


### A. Row counts and previews

In [ ]:
%%bash
set -e
cd lecture2_18h
wc -l data/values.csv
head -n 3 data/values.csv
tail -n 3 data/values.csv


### B. Introduce a bad row and detect it

In [ ]:
%%bash
set -e
cd lecture2_18h
cp data/values.csv data/values_bad.csv
echo "6,NOT_A_NUMBER" >> data/values_bad.csv

echo "Preview bad file:"
tail -n 3 data/values_bad.csv

echo "Find non-numeric entries in x column (simple grep):"
cut -d, -f2 data/values_bad.csv | grep -vE '^[0-9]+$' || true


### Reflection (Data hygiene)
Write 4–6 lines in `notes/lecture2_reflection.md`:

- Why is early data inspection a scientific responsibility?  
- Give one example of how a bad row could destroy a physics conclusion.  


In [ ]:
%%bash
set -e
cd lecture2_18h
cat > notes/lecture2_reflection.md << 'EOF'
# Lecture 2 Reflection (Data Hygiene)

- Why is early data inspection a scientific responsibility?

- Give one example of how a bad row could destroy a physics conclusion.

EOF
sed -n '1,120p' notes/lecture2_reflection.md


---
# Search & pattern matching with `grep` (diagnostics mindset)

In computational physics, logs are often huge.  
You need fast ways to extract:
- warnings
- instability indicators
- parameter settings
- convergence summaries

`grep` is your first diagnostic tool.


### A. Create a "dummy" simulation log

In [ ]:
%%bash
set -e
cd lecture2_18h
cat > results/sim.log << 'EOF'
INFO dt=0.1 N=50
INFO step=1  E=1.001
INFO step=2  E=1.003
WARNING step=3  energy_drift=high
INFO step=4  E=1.050
ERROR step=5  instability_detected
INFO done status=FAIL
EOF

cat results/sim.log


### B. Extract warnings and errors

In [ ]:
%%bash
set -e
cd lecture2_18h
echo "Warnings:"
grep "WARNING" results/sim.log || true
echo "Errors:"
grep "ERROR" results/sim.log || true


### C. Extract parameter lines

In [ ]:
%%bash
set -e
cd lecture2_18h
grep "^INFO dt=" results/sim.log


### Try it yourself (grep)
1) Extract all lines containing `E=`  
2) Count how many such lines exist  
3) Save them to `results/E_lines.txt`  


---
# Text processing for quick statistics: `cut`, `awk`, `sed`

When your dataset is large, you need quick summaries before Python.

Tools:
- `cut` for columns
- `awk` for computation (min/max/mean)
- `sed` for simple edits

We will use `awk` to compute min/max/mean of a column.


### A. Use awk to compute min/max/mean for x values

In [ ]:
%%bash
set -e
cd lecture2_18h
# Use values.csv (x is column 2), skip header
awk -F, 'NR>1 {x=$2; sum+=x; if(NR==2||x<min)min=x; if(NR==2||x>max)max=x} END{print "min=",min,"max=",max,"mean=",sum/(NR-1)}' data/values.csv


### B. Clean a file: remove a bad row (sed/grep approach)

We will remove lines containing NOT_A_NUMBER.


In [ ]:
%%bash
set -e
cd lecture2_18h
grep -v "NOT_A_NUMBER" data/values_bad.csv > data/values_clean.csv
tail -n 3 data/values_clean.csv


---
# Exit codes, `set -e`, and making scripts robust

Unix commands return **exit codes**:
- `0` = success
- non-zero = failure

Scientific scripts must fail loudly when something goes wrong.

We practice:
- checking exit codes
- using `set -euo pipefail`
- using traps for cleanup


### A. Check exit codes

In [ ]:
%%bash
set -e
cd lecture2_18h
ls data/values.csv
echo "exit code:" $?
ls data/does_not_exist.csv || true
echo "exit code (after failure):" $?


### B. A robust script template

In [ ]:
%%bash
set -e
cd lecture2_18h
cat > scripts/robust_template.sh << 'EOF'
#!/bin/bash
set -euo pipefail

log() { echo "$(date -Iseconds) | $*"; }

log "START robust_template"

# Example check: required file exists
FILE="data/values.csv"
if [[ ! -f "$FILE" ]]; then
  log "ERROR missing required file: $FILE"
  exit 1
fi

log "INFO found $FILE"
log "END robust_template status=OK"
EOF

chmod +x scripts/robust_template.sh
./scripts/robust_template.sh | tee results/robust_template.log


---
# Parameter sweeps and experiment bookkeeping

A core activity in computational physics is a **parameter sweep**:
- run the same model for many dt values
- compare stability and accuracy
- record outputs consistently

We will create:
- a parameter list
- a script that loops over parameters
- a runbook log entry per run


### A. Create a simple parameter list file

In [ ]:
%%bash
set -e
cd lecture2_18h
cat > data/params_dt.txt << 'EOF'
0.2
0.1
0.05
0.02
0.01
EOF

cat data/params_dt.txt


### B. Create a fake 'simulation' that prints diagnostics

In [ ]:
%%bash
set -e
cd lecture2_18h
cat > scripts/fake_sim.sh << 'EOF'
#!/bin/bash
set -euo pipefail
dt="$1"
N="$2"

echo "INFO dt=$dt N=$N"
# Fake diagnostics: energy drift scales with dt (toy)
drift=$(awk -v dt="$dt" 'BEGIN{print dt*dt*10}')
echo "INFO drift=$drift"
# Fake fail condition
awk -v d="$dt" 'BEGIN{exit (d>0.15)?1:0}'
echo "INFO status=OK"
EOF
chmod +x scripts/fake_sim.sh


### C. Sweep dt values and log results

In [ ]:
%%bash
set -e
cd lecture2_18h
N=100
OUT="results/sweep.log"
: > "$OUT"

while read -r dt; do
  ts=$(date -Iseconds)
  echo "$ts | RUN dt=$dt N=$N" >> "$OUT"
  if ./scripts/fake_sim.sh "$dt" "$N" >> "$OUT" 2>> "$OUT"; then
    echo "$ts | RESULT dt=$dt status=OK" >> "$OUT"
  else
    echo "$ts | RESULT dt=$dt status=FAIL" >> "$OUT"
  fi
  echo "----" >> "$OUT"
done < data/params_dt.txt

sed -n '1,120p' results/sweep.log


### Try it yourself (parameter sweeps)
1) Change the fail threshold in `fake_sim.sh`  
2) Re-run the sweep  
3) Explain how the sweep log changes  


---
# Plotting hooks: pass Unix results into Python

Unix excels at:
- organizing
- filtering
- summarizing

Python excels at:
- analysis
- visualization

We now connect them in a reproducible pipeline:
1) Unix produces a clean `results/sweep_summary.csv`
2) Python reads it and plots drift vs dt

This pattern is extremely common in computational physics projects.


### A. Extract sweep summary into a CSV using grep/awk

In [ ]:
%%bash
set -e
cd lecture2_18h
# Build a CSV: dt, drift, status
echo "dt,drift,status" > results/sweep_summary.csv

# Parse blocks: grab dt line, drift line, status line (toy parsing)
awk '
  /INFO dt=/ { 
    match($0, /dt=([0-9.]+)/, a); dt=a[1]; 
  }
  /INFO drift=/ {
    match($0, /drift=([0-9.]+)/, b); drift=b[1];
  }
  /RESULT dt=/ {
    match($0, /status=([A-Z]+)/, c); status=c[1];
    if (dt != "" && drift != "" && status != "") {
      print dt "," drift "," status
      dt=""; drift=""; status=""
    }
  }
' results/sweep.log >> results/sweep_summary.csv

cat results/sweep_summary.csv


### B. Python plotting hook (no styling, no custom colors)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("lecture2_18h/results/sweep_summary.csv")
df


In [ ]:
import numpy as np

# Convert numeric columns
df["dt"] = df["dt"].astype(float)
df["drift"] = df["drift"].astype(float)

# Plot (default matplotlib styling)
plt.figure()
plt.plot(df["dt"], df["drift"], marker="o")
plt.xlabel("dt")
plt.ylabel("drift (toy units)")
plt.title("Diagnostic: drift vs dt")
plt.gca().invert_xaxis()  # smaller dt to the right is common, but optional
plt.show()


### Try it yourself (plotting hooks)
1) Filter only rows where status == OK and plot those  
2) Add a second plot: log(drift) vs log(dt) (log-log)  
3) What relationship do you observe? (Write 2–3 sentences.)  


---
# <font color="blue">Mini-project: Build a reproducible “experiment runner”</font>

**Goal:** Create a reusable workflow for experiments.

Deliverables:
- `scripts/run_experiment.sh` that:
  - takes dt and N as inputs
  - runs the simulation (fake for now)
  - logs to `results/runbook.log`
  - updates `results/sweep_summary.csv`

Success criteria:
- If a run fails, it should be logged clearly.
- Running the script multiple times should append safely (not overwrite silently).

Start from the robust template patterns you learned earlier.


### Starter scaffold for the mini-project

In [ ]:
%%bash
set -e
cd lecture2_18h
cat > scripts/run_experiment.sh << 'EOF'
#!/bin/bash
set -euo pipefail

dt="${1:-}"
N="${2:-}"

if [[ -z "$dt" || -z "$N" ]]; then
  echo "USAGE: ./scripts/run_experiment.sh <dt> <N>"
  exit 2
fi

logfile="results/runbook.log"
sweepcsv="results/sweep_summary.csv"

ts=$(date -Iseconds)
echo "$ts | START dt=$dt N=$N" >> "$logfile"

# Run the simulation (toy)
if ./scripts/fake_sim.sh "$dt" "$N" > results/last_run.out 2> results/last_run.err; then
  status="OK"
else
  status="FAIL"
fi

# Extract drift from stdout (toy)
drift=$(grep "INFO drift=" results/last_run.out | awk -F= '{print $2}' | tr -d ' ')

echo "$ts | END dt=$dt N=$N status=$status drift=$drift" >> "$logfile"

# Ensure CSV has header
if [[ ! -f "$sweepcsv" ]]; then
  echo "dt,drift,status" > "$sweepcsv"
fi

echo "$dt,$drift,$status" >> "$sweepcsv"

echo "Saved:"
echo " - $logfile"
echo " - $sweepcsv"
EOF

chmod +x scripts/run_experiment.sh

# Quick demo run:
./scripts/run_experiment.sh 0.1 100
tail -n 5 results/runbook.log
tail -n 5 results/sweep_summary.csv


### Reflection (Mini-project)
Answer in `notes/mini_project_reflection.md`:
1) What makes your workflow reproducible?  
2) What would break reproducibility?  
3) What would you add before running on an HPC cluster?  


In [ ]:
%%bash
set -e
cd lecture2_18h
cat > notes/mini_project_reflection.md << 'EOF'
# Mini-project Reflection

1) What makes your workflow reproducible?

2) What would break reproducibility?

3) What would you add before running on an HPC cluster?

EOF
sed -n '1,120p' notes/mini_project_reflection.md


---
# Computational Physics Case Study Hook (for later weeks)

In later weeks, your “simulation” will become a real model:
- SHM (ODE)
- projectile motion with drag
- heat equation (PDE)
- random walk / diffusion (stochastic)

The Unix workflow remains the same:
- organize
- log
- diagnose
- summarize
- plot

**Key idea:** Tools change; workflow discipline stays.


---
# Mastery checklist (Lecture 2)

You are doing well if you can:
- Explain stdin/stdout/stderr in plain language  
- Redirect outputs safely and intentionally  
- Build pipelines that compute simple summaries  
- Inspect datasets before analysis  
- Use grep for log diagnostics  
- Use awk for quick stats  
- Write robust scripts (set -euo pipefail)  
- Run parameter sweeps with consistent logging  
- Export summaries for Python plotting hooks  

**If you can do these, you are ready for Week 3–4 Python fundamentals.**
